# ⚙️ Notebook 02 — Preprocessing Pipeline
## Human Intrusion Detection System

**GPU**: NVIDIA Quadro T2000 (4GB VRAM)  
**Objective**: Build and validate the full preprocessing pipeline:
1. COCO → YOLO format conversion (person-only)
2. Train/val/test split with stratification
3. Albumentations augmentation pipeline demo
4. Preprocessing for Quadro T2000 constraints
5. Frame extraction from video (MOT17 + VIRAT)
---

In [ ]:
import sys, os, json, shutil, random
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from tqdm.notebook import tqdm
import yaml

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Fix: Robust root detection (works from any Jupyter launch directory)
def _find_root():
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / 'src').is_dir() and (p / 'requirements.txt').exists():
            return p
    return Path.cwd()
ROOT = _find_root()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')
print(f'src/ found  : {(ROOT / "src").is_dir()}')
DATA_DIR   = ROOT / 'data'
RAW_DIR    = DATA_DIR / 'raw'
PROC_DIR   = DATA_DIR / 'processed'
COCO_DIR   = RAW_DIR  / 'coco'
MOT17_DIR  = RAW_DIR  / 'mot17'

PROC_DIR.mkdir(parents=True, exist_ok=True)

# Intrusion dataset output directories (YOLO structure)
for split in ['train', 'val', 'test']:
    (PROC_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (PROC_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

print(f'✅ Project root: {ROOT}')
print(f'✅ Output dir  : {PROC_DIR}')
print()
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 🔄 Section 1 — COCO → YOLO Format Conversion (Person Only)

In [ ]:
# COCO person class is category_id=1
# We map it to YOLO class 0 (our only class: person)
PERSON_CAT_ID = 1   # COCO
YOLO_CLASS_ID = 0   # Our person class

def convert_coco_to_yolo(ann_file: Path, img_src_dir: Path,
                          out_img_dir: Path, out_lbl_dir: Path,
                          min_area: float = 200.0,
                          min_size: int = 20) -> dict:
    """
    Convert COCO annotations to YOLO format, person class only.
    Copies images alongside labels.
    
    Filters:
        min_area  : minimum bbox area (pixels²) — removes tiny detections
        min_size  : minimum bbox width AND height in pixels
    """
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)
    
    print(f'Loading {ann_file.name} ...')
    with open(ann_file) as f:
        coco = json.load(f)
    
    img_info = {img['id']: img for img in coco['images']}
    
    # Build per-image annotations (person only)
    anns_by_img = defaultdict(list)
    filtered_small = 0
    for ann in coco['annotations']:
        if ann['category_id'] != PERSON_CAT_ID:
            continue
        bx, by, bw, bh = ann['bbox']
        if ann['area'] < min_area or bw < min_size or bh < min_size:
            filtered_small += 1
            continue
        if ann.get('iscrowd', 0):   # skip crowd annotations
            continue
        anns_by_img[ann['image_id']].append(ann)
    
    stats = {'converted': 0, 'skipped_no_ann': 0, 
             'filtered_small': filtered_small, 'total_boxes': 0}
    
    for img_id, anns in tqdm(anns_by_img.items(), desc='Converting'):
        info = img_info.get(img_id)
        if not info:
            continue
        
        src_path = img_src_dir / info['file_name']
        if not src_path.exists():
            stats['skipped_no_ann'] += 1
            continue
        
        W, H = info['width'], info['height']
        stem = src_path.stem
        
        # Build YOLO label lines
        lines = []
        for ann in anns:
            bx, by, bw, bh = ann['bbox']
            cx = (bx + bw / 2) / W
            cy = (by + bh / 2) / H
            nw = bw / W
            nh = bh / H
            # Clamp
            cx = max(0.001, min(0.999, cx))
            cy = max(0.001, min(0.999, cy))
            nw = max(0.001, min(0.999, nw))
            nh = max(0.001, min(0.999, nh))
            lines.append(f'{YOLO_CLASS_ID} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        
        if not lines:
            continue
        
        # Save label
        lbl_path = out_lbl_dir / f'{stem}.txt'
        with open(lbl_path, 'w') as f:
            f.write('\n'.join(lines))
        
        # Copy image
        dst_path = out_img_dir / src_path.name
        if not dst_path.exists():
            shutil.copy2(src_path, dst_path)
        
        stats['converted'] += 1
        stats['total_boxes'] += len(lines)
    
    return stats

# Run conversion
val_ann = COCO_DIR / 'annotations' / 'instances_val2017.json'

if val_ann.exists():
    stats = convert_coco_to_yolo(
        ann_file    = val_ann,
        img_src_dir = COCO_DIR / 'val2017',
        out_img_dir = PROC_DIR / 'images' / 'val',
        out_lbl_dir = PROC_DIR / 'labels' / 'val',
        min_area=200,
        min_size=20
    )
    print(f'\n✅ Conversion complete:')
    for k, v in stats.items():
        print(f'   {k:20s}: {v:,}')
else:
    print('⚠️  COCO annotations not found. Run download in Notebook 01 first.')
    print('   Showing conversion logic above for reference.')

In [ ]:
# Convert training set (larger — ~118K images)
train_ann = COCO_DIR / 'annotations' / 'instances_train2017.json'

if train_ann.exists():
    print('Converting COCO train2017 (this may take 5-10 minutes)...')
    stats_train = convert_coco_to_yolo(
        ann_file    = train_ann,
        img_src_dir = COCO_DIR / 'train2017',
        out_img_dir = PROC_DIR / 'images' / 'train',
        out_lbl_dir = PROC_DIR / 'labels' / 'train',
        min_area=200,
        min_size=20
    )
    print(f'\n✅ Train conversion:')
    for k, v in stats_train.items():
        print(f'   {k:20s}: {v:,}')
else:
    print('⚠️  train2017 annotations not found.')
    print('   Download: http://images.cocodataset.org/zips/train2017.zip (~18GB)')

In [ ]:
# ── Generate dataset.yaml for YOLO training ──────────────────
dataset_yaml = {
    'path'  : str(PROC_DIR.absolute()),
    'train' : 'images/train',
    'val'   : 'images/val',
    'nc'    : 1,           # person ONLY
    'names' : ['person'],
}

yaml_path = PROC_DIR / 'intrusion.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print('✅ Dataset YAML generated:')
print()
with open(yaml_path) as f:
    print(f.read())
print(f'Saved: {yaml_path}')

## 🎨 Section 2 — Augmentation Pipeline Demo

In [ ]:
# ── Define augmentation pipelines ────────────────────────────
IMG_SIZE = 640

train_transform = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE,
                  border_mode=cv2.BORDER_CONSTANT, value=(114, 114, 114)),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.4,
                       rotate_limit=10, p=0.7,
                       border_mode=cv2.BORDER_CONSTANT, value=(114,114,114)),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20),
        A.CLAHE(clip_limit=4.0),
    ], p=0.8),
    A.OneOf([
        A.GaussNoise(var_limit=(10, 50)),
        A.MotionBlur(blur_limit=7),
        A.GaussianBlur(blur_limit=5),
    ], p=0.4),
    A.CoarseDropout(max_holes=6, max_height=64, max_width=64,
                    fill_value=(114,114,114), p=0.3),
    A.RandomShadow(p=0.2),
    A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.3, p=0.15),
], bbox_params=A.BboxParams(format='yolo', label_fields=['labels'], min_visibility=0.3))

val_transform = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE,
                  border_mode=cv2.BORDER_CONSTANT, value=(114, 114, 114)),
], bbox_params=A.BboxParams(format='yolo', label_fields=['labels']))

print('✅ Augmentation pipelines defined:')
print(f'   Train transforms: {len(train_transform.transforms)} steps')
print(f'   Val transforms  : {len(val_transform.transforms)} steps')

In [ ]:
# ── Visualize augmentation on sample images ───────────────────
def load_random_sample(img_dir: Path, lbl_dir: Path):
    """Load a random image + its YOLO labels."""
    img_files = list(img_dir.glob('*.jpg'))
    if not img_files:
        return None, None, None
    img_path = random.choice(img_files)
    lbl_path = lbl_dir / (img_path.stem + '.txt')
    
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    bboxes, labels = [], []
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = list(map(float, line.split()))
            labels.append(int(parts[0]))
            bboxes.append(parts[1:5])  # cx, cy, w, h
    
    return img, bboxes, labels

def yolo_to_pixel(bbox, W, H):
    cx, cy, bw, bh = bbox
    x1 = int((cx - bw/2) * W)
    y1 = int((cy - bh/2) * H)
    x2 = int((cx + bw/2) * W)
    y2 = int((cy + bh/2) * H)
    return x1, y1, x2, y2

def draw_boxes(ax, img, bboxes, title='', color='#00FF88'):
    ax.imshow(img)
    H, W = img.shape[:2]
    for bbox in bboxes:
        x1, y1, x2, y2 = yolo_to_pixel(bbox, W, H)
        rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

# Load sample
val_img_dir = PROC_DIR / 'images' / 'val'
val_lbl_dir = PROC_DIR / 'labels' / 'val'

img_sample, bboxes_orig, labels_orig = load_random_sample(val_img_dir, val_lbl_dir)

if img_sample is not None:
    # Apply augmentations 6 times
    aug_results = []
    for _ in range(6):
        res = train_transform(image=img_sample.copy(), 
                              bboxes=bboxes_orig, 
                              labels=labels_orig)
        aug_results.append((res['image'], res['bboxes']))
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 9))
    fig.suptitle('Augmentation Pipeline — YOLOv8s Training Transforms', 
                 fontsize=13, fontweight='bold')
    
    # Original
    val_res = val_transform(image=img_sample, bboxes=bboxes_orig, labels=labels_orig)
    draw_boxes(axes[0,0], val_res['image'], val_res['bboxes'], 
               f'Original (Letterboxed 640×640)\n{len(bboxes_orig)} persons', '#FFD700')
    
    for i, (aug_img, aug_bboxes) in enumerate(aug_results):
        row, col = (i+1) // 4, (i+1) % 4
        aug_names = ['Flip+Brightness', 'Scale+Blur', 
                     'Dropout+Shadow', 'Rotate+Noise',
                     'Fog+CLAHE', 'ShiftScale+Contrast']
        draw_boxes(axes[row, col], aug_img, aug_bboxes,
                   f'{aug_names[i]}\n{len(aug_bboxes)} persons (visible)', '#00FF88')
    
    axes[1, 3].axis('off')  # hide last if uneven
    plt.tight_layout()
    plt.savefig('preprocessing_augmentation_demo.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('✅ Augmentation visualization saved.')
else:
    print('⚠️  No processed images found. Run COCO conversion first.')
    print('   Augmentation pipeline is still defined and ready to use.')

## 🎬 Section 3 — Video Frame Extraction (MOT17 + VIRAT)

In [ ]:
def extract_frames_from_sequence(seq_dir: Path, output_dir: Path,
                                  sample_every: int = 3,
                                  max_frames: int = None) -> int:
    """
    Extract frames from a MOT17 sequence folder.
    MOT17 stores frames as JPEG images in img1/
    
    Returns: number of frames extracted
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    img1_dir = seq_dir / 'img1'
    gt_file  = seq_dir / 'gt' / 'gt.txt'
    
    if not img1_dir.exists():
        print(f'  No img1/ found in {seq_dir.name}')
        return 0
    
    frame_files = sorted(img1_dir.glob('*.jpg'))
    
    # Parse ground truth into frame-level dicts
    gt_data = defaultdict(list)
    if gt_file.exists():
        for line in gt_file.read_text().strip().splitlines():
            p = line.split(',')
            if len(p) < 7 or p[6] != '1': continue  # skip inactive
            fid = int(p[0])
            x, y, w, h = float(p[2]), float(p[3]), float(p[4]), float(p[5])
            gt_data[fid].append((x, y, w, h))
    
    lbl_out = output_dir.parent.parent / 'labels' / output_dir.parent.name / output_dir.name
    lbl_out.mkdir(parents=True, exist_ok=True)
    
    saved = 0
    for i, ff in enumerate(frame_files):
        if i % sample_every != 0:
            continue
        if max_frames and saved >= max_frames:
            break
        
        frame_id = int(ff.stem)
        img = cv2.imread(str(ff))
        if img is None: continue
        H, W = img.shape[:2]
        
        dst_img = output_dir / f'{seq_dir.name}_{ff.stem}.jpg'
        cv2.imwrite(str(dst_img), img)
        
        # Write YOLO label
        lines = []
        for (x, y, w, h) in gt_data.get(frame_id, []):
            if w < 15 or h < 15: continue
            cx = (x + w/2) / W
            cy = (y + h/2) / H
            nw = w / W
            nh = h / H
            cx = max(0.001, min(0.999, cx))
            cy = max(0.001, min(0.999, cy))
            nw = max(0.001, min(0.999, nw))
            nh = max(0.001, min(0.999, nh))
            lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        
        if lines:
            lbl_path = lbl_out / f'{seq_dir.name}_{ff.stem}.txt'
            lbl_path.write_text('\n'.join(lines))
            saved += 1
    
    return saved

# Run extraction if MOT17 exists
mot17_train = MOT17_DIR / 'train'

if mot17_train.exists():
    sequences = sorted([d for d in mot17_train.iterdir() if d.is_dir()])
    print(f'Found {len(sequences)} MOT17 sequences.')
    
    total_saved = 0
    for seq in tqdm(sequences, desc='Extracting MOT17 frames'):
        out_dir = PROC_DIR / 'images' / 'train'
        n = extract_frames_from_sequence(
            seq_dir=seq,
            output_dir=out_dir,
            sample_every=5,   # take 1 frame every 5
            max_frames=500,   # cap per sequence
        )
        total_saved += n
        print(f'  {seq.name}: {n} frames extracted')
    
    print(f'\n✅ Total MOT17 frames added to training set: {total_saved:,}')
else:
    print('⚠️  MOT17 not found.')
    print('   Download: https://motchallenge.net/data/MOT17.zip')

In [ ]:
# ── GPU Memory Budget Analysis for Quadro T2000 ──────────────
import math

print('='*60)
print('  💾 GPU Memory Budget — NVIDIA Quadro T2000')
print('='*60)

VRAM_MB     = 4096       # T2000 total VRAM
IMG_SIZE    = 640
BATCH       = 8
CHANNELS    = 3
BYTES_FP16  = 2          # FP16 = 2 bytes per element
BYTES_FP32  = 4          # FP32 = 4 bytes per element

# Input tensor: batch × channels × H × W
input_mb_fp16 = (BATCH * CHANNELS * IMG_SIZE * IMG_SIZE * BYTES_FP16) / (1024**2)
input_mb_fp32 = (BATCH * CHANNELS * IMG_SIZE * IMG_SIZE * BYTES_FP32) / (1024**2)

# YOLOv8s parameter count ≈ 11M params
model_params = 11_166_560   # YOLOv8s
model_mb_fp16 = (model_params * BYTES_FP16) / (1024**2)
model_mb_fp32 = (model_params * BYTES_FP32) / (1024**2)

# Gradient memory ≈ same as model (fp32 gradients)
grad_mb = model_mb_fp32

# Activations estimate (feature maps through backbone) ≈ 3× input
activations_mb = input_mb_fp16 * 3

total_mb = input_mb_fp16 + model_mb_fp16 + grad_mb + activations_mb

print(f'\n  Configuration:')
print(f'    Model      : YOLOv8s ({model_params:,} params)')
print(f'    Image size : {IMG_SIZE}×{IMG_SIZE}')
print(f'    Batch size : {BATCH}')
print(f'    Precision  : FP16 (inputs + model weights)')
print()
print(f'  Memory Breakdown:')
print(f'    Input batch   : {input_mb_fp16:.1f} MB  (FP16)')
print(f'    Model weights : {model_mb_fp16:.1f} MB  (FP16)')
print(f'    Gradients     : {grad_mb:.1f} MB  (FP32)')
print(f'    Activations   : {activations_mb:.1f} MB  (FP16 est.)')
print(f'    ─────────────────────────────')
print(f'    Total estimate: {total_mb:.0f} MB')
print(f'    Available VRAM: {VRAM_MB} MB')
print(f'    Headroom      : {VRAM_MB - total_mb:.0f} MB')
print()

if total_mb < VRAM_MB * 0.85:
    print(f'  ✅ Configuration fits comfortably in 4GB VRAM!')
    print(f'     Safety margin: {(VRAM_MB - total_mb)/VRAM_MB*100:.0f}%')
elif total_mb < VRAM_MB:
    print(f'  ⚠️  Configuration fits but is tight. Monitor with nvidia-smi.')
else:
    print(f'  ❌ Reduce batch_size to avoid OOM!')
    safe_batch = max(1, int(BATCH * VRAM_MB / total_mb * 0.8))
    print(f'     Recommended batch_size: {safe_batch}')

print()
print('  📝 Optimal Settings for Quadro T2000:')
print('     model   = yolov8s    (11M params, not yolov8m=25M)')
print('     batch   = 8          (not 16)')
print('     imgsz   = 640        (ok, standard)')
print('     half    = True       (FP16 essential)')
print('     cache   = False      (save RAM for VRAM)')
print('     workers = 4          (not 8)')

In [ ]:
# ── Final dataset statistics ──────────────────────────────────
print('='*60)
print('  📊 Processed Dataset Statistics')
print('='*60)

for split in ['train', 'val', 'test']:
    img_dir = PROC_DIR / 'images' / split
    lbl_dir = PROC_DIR / 'labels' / split
    
    n_imgs = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    n_lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    
    # Count total person boxes
    total_boxes = 0
    if lbl_dir.exists():
        for lf in lbl_dir.glob('*.txt'):
            lines = lf.read_text().strip().splitlines()
            total_boxes += len([l for l in lines if l.strip()])
    
    avg_boxes = total_boxes / max(n_lbls, 1)
    print(f'  {split:6s}: {n_imgs:6,} images | {n_lbls:6,} labels | '
          f'{total_boxes:8,} boxes | {avg_boxes:.1f} persons/img')

print()
print('  ➡️  Next: Run Notebook 03 — Model Training')